# AIE Day2 発展：用語集SFTと検索比較

58語・116QA、train58/valid58。Colab T4、配布の固定ライブラリを使う。4つのコードセルを上から実行する。APIキー不要。20エポック160ステップを学習し、valid loss最小のチェックポイントを選ぶ。

初回5エポックの失敗後の探索的追試で、元42問に加えて追試前に固定した追加12問も評価する。結果は `/content/aie_day2_advanced/results20/` に保存する。定義を入力に渡さず、通常の文章を出力する。語群一致は正確性の代理指標である。

この再実行版は長さ計測を `input_ids` で修正済み。実測当時の元ソース・初回失敗・全出力・計測修正の証跡は提出ZIPの `lectures/02/advanced/runs/` と `REPORT.md` を参照。再実行は環境差により結果が変わりうる。結果を回収したらランタイムを解放する。


In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

In [ ]:
from pathlib import Path
import base64, io, zipfile, hashlib, sys, json
ROOT = Path('/content/aie_day2_advanced')
ROOT.mkdir(exist_ok=True)
# 公開するソース/用語集/生成データ/評価42問を同梱。ZIP内の各ファイルをそのまま展開する。
PAYLOAD = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(ROOT)
sys.path.insert(0, str(ROOT))
manifest = json.loads((ROOT / 'data/manifest.json').read_text())
for name, expected in manifest['sha256'].items():
    assert hashlib.sha256((ROOT / 'data' / name).read_bytes()).hexdigest() == expected
print('INPUTS_VERIFIED', {k: manifest[k] for k in ['terms','generated_rows','train','valid','test']})


In [ ]:
%run /content/aie_day2_advanced/run_experiment.py


In [ ]:
from pathlib import Path
import json, hashlib
RESULTS = Path('/content/aie_day2_advanced/results20')
names = ['result.json', 'eval_before.json', 'eval_after.json', 'eval_search.json', 'token_lengths.json', 'run_manifest.json', 'eval_audit_before.json', 'eval_audit_after.json', 'eval_audit_search.json']
if not (RESULTS / 'result.json').exists():
    print('RESULTS_NOT_READY', [p.name for p in RESULTS.glob('*.json')])
else:
    payload = {name: (RESULTS / name).read_text(encoding='utf-8') for name in names}
    print('BEGIN_ADVANCED_RESULTS')
    print(json.dumps(payload, ensure_ascii=False))
    print('END_ADVANCED_RESULTS')
    print(json.dumps({name: hashlib.sha256((RESULTS / name).read_bytes()).hexdigest() for name in names}))
    result = json.loads(payload['result.json'])
    print('AIE Day2 Advanced | Glossary SFT vs search | Tesla T4')
    print('Training:', round(result['training_seconds'], 2), 'seconds /', result['global_step'], 'steps')
    for label in ['before','after','search']:
        s = result[label]
        print(label, 'known:', round(s['known_concept_rate']*100,2), '% / unknown:', round(s['unknown_abstention_rate']*100,2), '%')
        print({kind: str(v['correct'])+'/'+str(v['n']) for kind,v in s['by_kind'].items()})
